In [11]:
# Install dependencies if needed
import subprocess, sys
for pkg in ["plotly", "pandas"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

In [13]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Path to output directory from mainSmoother — adjust if needed
OUTPUT_DIR = Path("../build/legged_estimator_outputs")

VARIANTS = [
    "fixed_lag_combined_bias",
]

LABELS = {
    "fixed_lag_combined_bias": "Fixed-lag (combined bias)",
}

COLORS = {
    "fixed_lag_combined_bias": "#d62728",
}

LINE_STYLES = {
    "fixed_lag_combined_bias": "dot",
}

# Load trajectory CSVs — columns: kind,timestamp_s,x,y,z,qw,qx,qy,qz,roll,pitch,yaw,vx,vy,vz
DISPLAY_RESULTS = {}
for variant in VARIANTS:
    csv_path = OUTPUT_DIR / f"{variant}_trajectory.csv"
    if not csv_path.exists():
        print(f"Warning: {csv_path} not found, skipping")
        continue
    df = pd.read_csv(csv_path)
    DISPLAY_RESULTS[variant] = {"trajectory": df.to_dict(orient="records")}

if not DISPLAY_RESULTS:
    raise RuntimeError(f"No trajectory CSVs found in {OUTPUT_DIR.resolve()}")

# Use the earliest timestamp as the display start
DISPLAY_START_TIME_S = min(
    pd.DataFrame(result["trajectory"])["timestamp_s"].iloc[0]
    for result in DISPLAY_RESULTS.values()
)

print(f"Loaded variants: {list(DISPLAY_RESULTS.keys())}")
print(f"Display start time: {DISPLAY_START_TIME_S:.3f} s")
for variant, result in DISPLAY_RESULTS.items():
    print(f"  {variant}: {len(result['trajectory'])} trajectory rows")

Loaded variants: ['fixed_lag_combined_bias']
Display start time: 0.020 s
  fixed_lag_combined_bias: 281 trajectory rows


In [14]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Forward position", "Estimated height"))
for variant, result in DISPLAY_RESULTS.items():
    trajectory = result["trajectory"]
    times = np.array([row["timestamp_s"] for row in trajectory])
    xs = np.array([row["x"] for row in trajectory])
    zs = np.array([row["z"] for row in trajectory])
    line = dict(color=COLORS[variant], width=3, dash=LINE_STYLES[variant])
    fig.add_trace(go.Scatter(x=times, y=xs, mode="lines", name=LABELS[variant], line=line, legendgroup=variant), row=1, col=1)
    fig.add_trace(go.Scatter(x=times, y=zs, mode="lines", name=LABELS[variant], line=line, legendgroup=variant, showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="time [s]", row=1, col=1)
fig.update_xaxes(title_text="time [s]", row=1, col=2)
fig.update_yaxes(title_text="x [m]", row=1, col=1)
fig.update_yaxes(title_text="z [m]", row=1, col=2)
fig.update_layout(height=420, width=1100, template="plotly_white", legend=dict(x=0.01, y=0.99))
fig.show()


In [15]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Roll", "Pitch"))
for variant, result in DISPLAY_RESULTS.items():
    trajectory = result["trajectory"]
    times = np.array([row["timestamp_s"] for row in trajectory])
    rolls = np.rad2deg([row["roll"] for row in trajectory])
    pitches = np.rad2deg([row["pitch"] for row in trajectory])
    line = dict(color=COLORS[variant], width=3, dash=LINE_STYLES[variant])
    fig.add_trace(go.Scatter(x=times, y=rolls, mode="lines", name=LABELS[variant], line=line, legendgroup=variant), row=1, col=1)
    fig.add_trace(go.Scatter(x=times, y=pitches, mode="lines", name=LABELS[variant], line=line, legendgroup=variant, showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="time [s]", row=1, col=1)
fig.update_xaxes(title_text="time [s]", row=1, col=2)
fig.update_yaxes(title_text="degrees", row=1, col=1)
fig.update_yaxes(title_text="degrees", row=1, col=2)
fig.update_layout(height=420, width=1100, template="plotly_white", legend=dict(x=0.01, y=0.99))
fig.show()


In [16]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("v_x", "v_y", "v_z"))
velocity_components = (
    ("vx", "v_x [m/s]"),
    ("vy", "v_y [m/s]"),
    ("vz", "v_z [m/s]"),
)
for col, (field, axis_label) in enumerate(velocity_components, start=1):
    for variant, result in DISPLAY_RESULTS.items():
        trajectory = result["trajectory"]
        times = np.array([row["timestamp_s"] for row in trajectory])
        values = np.array([row[field] for row in trajectory])
        fig.add_trace(
            go.Scatter(
                x=times,
                y=values,
                mode="lines",
                name=LABELS[variant],
                line=dict(color=COLORS[variant], width=3, dash=LINE_STYLES[variant]),
                legendgroup=variant,
                showlegend=(col == 1),
            ),
            row=1,
            col=col,
        )
    fig.update_xaxes(title_text="time [s]", row=1, col=col)
    fig.update_yaxes(title_text=axis_label, row=1, col=col)
fig.update_layout(height=420, width=1200, template="plotly_white", legend=dict(x=0.01, y=0.99))
fig.show()
